In [3]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

In [4]:
import torch
import requests
from pathlib import Path
import subprocess
from time import perf_counter_ns
from datetime import timedelta

from joblib import Parallel, delayed, cpu_count
from torchcodec.decoders import VideoDecoder

def bench(f, *args, num_exp=3, warmup=1, **kwargs):
    """Benchmark a function by running it multiple times and measuring execution time"""
    for _ in range(warmup): f(*args, **kwargs)

    times=[]
    for _ in range(num_exp):
        start=perf_counter_ns()
        result=f(*args, **kwargs)
        end=perf_counter_ns()
        times.append(end-start)
    return torch.tensor(times).float(), result

def report_stats(times, unit='s'):
    """Report median and standard deviation of benchmark times"""
    mul={"ns":1, # nanoseconds
        "µs": 1e-3, # microseconds
        "ms": 1e-6, # milliseconds
        "s":1e-9, # seconds
        }[unit]
    times=times*mul
    std=times.std().item()
    med=times.median().item()
    print(f'median: ={med:.2f} +- {std:.2f} {unit}')
    return med

def split_indices(indices:list[int], num_chunks:int)->list[list[int]]:
    """Split a list of indices into approximately equal chunks"""
    chunk_size=len(indices)//num_chunks
    chunks=[]
    for i in range(num_chunks-1):
        chunks.append(indices[i*chunk_size:(i+1)*chunk_size])
    # Last chunk may be slightly larger
    chunks.append(indices[(num_chunks-1)*chunk_size:])
    return chunks

def generate_long_video(temp_dir:str):
    # Video source: https://www.pexels.com/video/dog-eating-854132/
    # License: CC0. Author: Coverr.
    url = "https://videos.pexels.com/video-files/854132/854132-sd_640_360_25fps.mp4"
    response=requests.get(url, headers={'User-Agent':''})
    if response.status_code!=200: raise RuntimeError(f'Failed to download video. {response.status_code=}')

    short_video_path=Path(temp_dir)/'short_video.mp4'
    with open(short_video_path, 'wb') as f: 
        for chunk in response.iter_content(): f.write(chunk)
    # Create a longer video by repeating the short one 50 times
    long_video_path=Path(temp_dir)/"long_video.mp4"
    ffmpeg_command=[
        "ffmpeg", "-y",
        "-stream_loop", "49", # repeat video 50 times
        "-i", str(short_video_path),
        "-c", "copy",
        str(long_video_path)
    ]
    subprocess.run(ffmpeg_command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return short_video_path,long_video_path

temp_dir=Path('D:/results/temp')
temp_dir.mkdir(parents=True, exist_ok=True)
short_video_path, long_video_path=generate_long_video(temp_dir)

decoder=VideoDecoder(long_video_path, seek_mode='approximate')
metadata=decoder.metadata

short_duration=timedelta(seconds=VideoDecoder(short_video_path).metadata.duration_seconds)
long_duration=timedelta(seconds=metadata.duration_seconds)
print(f"Original video duration: {int(short_duration.total_seconds()//60)} m {int(short_duration.total_seconds()%60):02d} s")
print(f"Long video duration: {int(long_duration.total_seconds()//60)} m {int(long_duration.total_seconds()%60):02d} s")
print(f"Video resolution: {metadata.width}x{metadata.height}")
print(f"Average FPS: {metadata.average_fps:.1f}")
print(f"Total frames: {metadata.num_frames}")

Original video duration: 0 m 13 s
Long video duration: 11 m 30 s
Video resolution: 640x360
Average FPS: 25.0
Total frames: 17250


In [5]:
TARGET_FPS=2
step=max(1, round(metadata.average_fps/TARGET_FPS)) # downsample FPS by 2
print('step ', step)
all_indices=list(range(0, metadata.num_frames, step))
print('all_indices ', len(all_indices))

print(f"Sampling 1 frame every {TARGET_FPS} seconds")
print(f"We'll skip every {step} frames")
print(f"Total frames to decode: {len(all_indices)}")

step  12
all_indices  1438
Sampling 1 frame every 2 seconds
We'll skip every 12 frames
Total frames to decode: 1438


### Method 1: Sequential decoding (baseline)

In [8]:
def decode_sequentially(indices:list[int], video_path=long_video_path):
    """Decode frames sequentially using a single decoder instance"""
    decoder=VideoDecoder(video_path, seek_mode="approximate")
    return decoder.get_frames_at(indices)

times, result_sequential=bench(decode_sequentially, all_indices)
sequential_time=report_stats(times, unit='s')

median: =5.51 +- 0.01 s


### Method 2: FFmpeg-based parallelism

In [9]:
def decode_with_ffmpeg_parallelism(indices:list[int], num_threads:int, video_path=long_video_path):
    """Decode frames using FFmpeg's internal threading"""
    decoder=VideoDecoder(video_path, num_ffmpeg_threads=num_threads, seek_mode="approximate")
    return decoder.get_frames_at(indices)
NUM_CPUS=cpu_count()
print('Number of CPUs: ', NUM_CPUS )

times, result_ffmpeg=bench(decode_with_ffmpeg_parallelism, all_indices, num_threads=NUM_CPUS)
ffmpeg_time=report_stats(times, unit='s')
speedup=sequential_time/ffmpeg_time
print(f'Speedup compared to sequential: {speedup:.2f} x with {NUM_CPUS} FFmpeg threads')

Number of CPUs:  32
median: =3.61 +- 0.01 s
Speedup compared to sequential: 1.53 x with 32 FFmpeg threads


### Method 3: multiprocessing

In [10]:
def decode_with_multiprocessing(indices:list[int], num_processes:int, video_path=long_video_path):
    """Decode frames using multiprocessing with joblib"""
    chunks=split_indices(indices, num_chunks=num_processes)
    # loky is a multi-processing backend for joblib: https://github.com/joblib/loky
    results=Parallel(n_jobs=num_processes, backend='loky', verbose=0)(delayed(decode_sequentially)(chunk, video_path) for chunk in chunks)
    return torch.cat([frame_batch.data for frame_batch in results], dim=0)

times, result_multiprocessing=bench(decode_with_multiprocessing, all_indices, num_processes=NUM_CPUS)
multiprocessing_time=report_stats(times, unit='s')
speedup=sequential_time/multiprocessing_time
print(f'Speedup compared to sequential: {speedup:.2f}x with {NUM_CPUS} processes')

median: =2.05 +- 0.11 s
Speedup compared to sequential: 2.68x with 32 processes


### Method 4: Joblib multithreading

In [11]:
def decode_with_multithreading(indices:list[int], num_threads:int, video_path=long_video_path):
    """Decode frames using multiple threads with joblib"""
    chunks=split_indices(indices, num_chunks=num_threads)
    results=Parallel(n_jobs=num_threads, prefer='threads', verbose=0)(delayed(decode_sequentially)(chunk, video_path) for chunk in chunks)
    # Concatenate results from all threads
    return torch.cat([frame_batch.data for frame_batch in results], dim=0)

times, result_multithreading=bench(decode_with_multithreading, all_indices, num_threads=NUM_CPUS)
multithreading_time=report_stats(times, unit='s')
speedup=sequential_time/multithreading_time
print(f'Speedup compared to sequential: {speedup:.2f}x with {NUM_CPUS} threads')

median: =0.71 +- 0.03 s
Speedup compared to sequential: 7.76x with 32 threads


In [15]:
result_multithreading.shape, result_ffmpeg.data.shape, result_sequential.data.shape

(torch.Size([1438, 3, 360, 640]),
 torch.Size([1438, 3, 360, 640]),
 torch.Size([1438, 3, 360, 640]))

In [16]:
result_ffmpeg

FrameBatch:
  data (shape): torch.Size([1438, 3, 360, 640])
  pts_seconds: tensor([0.0000e+00, 4.8000e-01, 9.6000e-01,  ..., 6.8880e+02, 6.8928e+02,
        6.8976e+02], dtype=torch.float64)
  duration_seconds: tensor([0.0400, 0.0400, 0.0400,  ..., 0.0400, 0.0400, 0.0400],
       dtype=torch.float64)

### Validation and correctness check

In [17]:
torch.testing.assert_close(result_sequential.data, result_ffmpeg.data, atol=0, rtol=0)
torch.testing.assert_close(result_sequential.data, result_multiprocessing, atol=0, rtol=0)
torch.testing.assert_close(result_sequential.data, result_multithreading, atol=0, rtol=0)
print('All good!')

All good!
